In [1]:
# STANDIN A0
# Load the data, the split and the shared helpers (parts/00_setup.ipynb). Dropped from the hand-in
# notebook, where the setup cells come first.
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project
Data folder: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project/UNSW-NB15
score() and score_proba() ready


Rows with an infinite or missing value: 0
Loaded 447,915 flows x 76 features


Raw rows:                      447,915
Exact duplicates removed:      141,742
Conflicting rows removed:        1,068
Clean rows (all are used):     305,105


Dropped 9 constant columns: Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, URG Flag Count, CWR Flag Count, ECE Flag Count, Fwd Bytes/Bulk Avg, Fwd Packet/Bulk Avg, Fwd Bulk Rate Avg
67 features: 65 continuous, 2 0/1 columns (Fwd PSH Flags, RST Flag Count)
TUNE_IDX: 36,612 training rows for settings searches
Setup complete: 22 shared names ready in 70 s


In [2]:
# STANDIN B2
# The Part B black box with the settings chosen on validation in B2 (parts/10_baseline_models.ipynb).
# Dropped from the hand-in notebook, where the real B2 cell comes first.
from sklearn.ensemble import HistGradientBoostingClassifier

BOOST_SETTINGS = {"learning_rate": 0.1, "max_leaf_nodes": 31, "random_state": SEED}
def make_boost():
    return HistGradientBoostingClassifier(**BOOST_SETTINGS)

boost = make_boost().fit(X_train, y_train)

<!-- STEP G1 -->
## Part G: adaptability, a new attack the model has never seen

Every test so far used attacks of the same kinds as in training. Real attackers change their methods, and
a model that scored well last month can quietly start missing a new kind of attack. Part G imitates
that on our own data.

**Step G1: hide one attack type.** We remove **every Fuzzers flow from the training set** (and only from
the training set) and train the black box again, with the same settings as in B2 (`make_boost()`). This
model, `boost_hidden`, has never seen a Fuzzers flow. Fuzzing means sending large amounts of random or
malformed data to make a program crash or misbehave.

**Why Fuzzers.** It is the third-largest attack type (15,606 training flows), and in a trial on the
**validation** set, before Part G was written, hiding Fuzzers lowered its recall the most of all attack
types (from 0.989 to 0.696; Exploits, Reconnaissance, DoS, Generic and Shellcode stayed at 0.97 or more).
The choice used no test data.

**What stays where.**
- The validation and test sets are unchanged: they still contain their Fuzzers flows (5,202 each), so we
  can measure how well the model recognises the new attack.
- The removed training flows are kept aside as `X_new, y_new`: the pool of flows an analyst could label in
  step G3, when the model is adapted. **Test-set Fuzzers flows are never used for training.**

In [3]:
# STEP G1
HIDDEN = "Fuzzers"
keep = (type_train != HIDDEN).to_numpy()

X_seen, y_seen = X_train[keep], y_train[keep]              # training set without the hidden type
X_new, y_new = X_train[~keep], y_train[~keep]              # the hidden flows: G3 may label some of them

assert len(X_new) == 15_606 and (y_new == 1).all()
assert (type_train[keep] != HIDDEN).all()
assert int((type_test == HIDDEN).sum()) == 5_202 and int((type_val == HIDDEN).sum()) == 5_202
assert not X_new.index.isin(X_test.index).any() and not X_new.index.isin(X_val.index).any()

boost_hidden = make_boost().fit(X_seen, y_seen)

g1 = pd.DataFrame({"training flows": [len(X_train), len(X_seen)],
                   "of which attacks": [int(y_train.sum()), int(y_seen.sum())],
                   "attack types": [type_train[y_train == 1].nunique(), type_train[keep][y_seen == 1].nunique()],
                   "trees": [boost.n_iter_, boost_hidden.n_iter_]},
                  index=["boost (Part B, all types)", f"boost_hidden (without {HIDDEN})"])
g1.to_csv(TABLES / "G1_hidden_training.csv")
print(f"Removed {len(X_new):,} {HIDDEN} flows from training; they stay in validation and test.")
g1

Removed 15,606 Fuzzers flows from training; they stay in validation and test.


,training flows,of which attacks,attack types,trees
"boost (Part B, all types)",183063,48154,9,100
boost_hidden (without Fuzzers),167457,32548,8,100


<!-- STEP G1 -->
### What G1 set up

| Model | Training flows | Attacks among them | Attack types | Trees |
|---|---:|---:|---:|---:|
| `boost` (Part B, all types) | 183,063 | 48,154 | 9 | 100 |
| `boost_hidden` (without Fuzzers) | 167,457 | 32,548 | 8 | 100 |

The 15,606 Fuzzers flows (a third of all training attacks) are gone from the training data; the model
otherwise has exactly the same settings. Step G2 measures on the test set what this costs: recall on all
attacks, and recall on the 5,202 Fuzzers flows alone, which this model has never seen.

<!-- STEP G2 -->
## Part G: what hiding Fuzzers costs

On the **test** set, the Part B model (`boost`, trained with Fuzzers) and `boost_hidden` (trained
without) are compared on:

- **overall recall**: the share of all 16,052 test attacks caught;
- **recall on Fuzzers**: the share of the 5,202 Fuzzers test flows caught, the attack `boost_hidden` has
  never seen;
- recall on the other attack types, the false alarm rate (FAR) and macro-F1.

Two extra views help explain the result and prepare G4 (how to notice a new attack in practice):
how sure each model is about the Fuzzers flows (the share with an attack probability between 0.1 and
0.9), and recall on Fuzzers split by whether the flow carries the attacker fingerprint from D1
(`Fwd Seg Size Min` = 20 and `FWD Init Win Bytes` = 16,383).

In [4]:
# STEP G2
is_hidden = (type_test == HIDDEN).to_numpy()
is_attack = (y_test == 1).to_numpy()
fingerprint = ((X_test["Fwd Seg Size Min"] == 20) & (X_test["FWD Init Win Bytes"] == 16383)).to_numpy()


def hidden_report(model):
    """The Part G numbers for one model on the test set."""
    p = model.predict_proba(X_test)[:, 1]
    pred = (p >= 0.5).astype(int)
    s = score_proba(y_test, p)
    return {"overall_recall": s["recall"],
            "fuzzers_recall": pred[is_hidden].mean(),
            "other_attacks_recall": pred[is_attack & ~is_hidden].mean(),
            "far": s["far"], "macro_f1": s["macro_f1"],
            "fuzzers_uncertain_share": ((p[is_hidden] > 0.1) & (p[is_hidden] < 0.9)).mean(),
            "fuzzers_recall_with_fingerprint": pred[is_hidden & fingerprint].mean(),
            "fuzzers_recall_without_fingerprint": pred[is_hidden & ~fingerprint].mean()}


g2 = pd.DataFrame({"boost (saw Fuzzers)": hidden_report(boost),
                   "boost_hidden (never saw Fuzzers)": hidden_report(boost_hidden)}).T
g2.loc["difference"] = g2.iloc[1] - g2.iloc[0]
g2.index.name = "model (test set)"
g2.to_csv(TABLES / "G2_hidden.csv", float_format="%.4f")
print(f"Fuzzers test flows: {is_hidden.sum():,}; with the attacker fingerprint: "
      f"{int((is_hidden & fingerprint).sum()):,} ({(is_hidden & fingerprint).sum() / is_hidden.sum():.1%})")
g2.round(4)

Fuzzers test flows: 5,202; with the attacker fingerprint: 4,859 (93.4%)


,overall_recall,fuzzers_recall,other_attacks_recall,far,macro_f1,fuzzers_uncertain_share,fuzzers_recall_with_fingerprint,fuzzers_recall_without_fingerprint
model (test set),,,,,,,,
boost (saw Fuzzers),0.9923,0.9875,0.9946,0.0276,0.9718,0.4314,0.9866,1.0000
boost_hidden (never saw Fuzzers),0.8880,0.6876,0.9841,0.0189,0.9429,0.8239,0.6719,0.9096
difference,-0.1043,-0.2999,-0.0105,-0.0087,-0.0289,0.3925,-0.3147,-0.0904


<!-- STEP G2 -->
### What the hidden attack costs

| Test set | `boost` (saw Fuzzers) | `boost_hidden` (never saw Fuzzers) | Difference |
|---|---:|---:|---:|
| **Recall on Fuzzers** (5,202 flows) | **0.988** | **0.688** | **−0.300** |
| Overall recall (16,052 attacks) | 0.992 | 0.888 | −0.104 |
| Recall on the other attack types | 0.995 | 0.984 | −0.010 |
| False alarm rate (FAR) | 0.028 | 0.019 | −0.009 |
| Macro-F1 | 0.972 | 0.943 | −0.029 |
| Fuzzers flows with attack probability between 0.1 and 0.9 | 43% | 82% | +39 points |

**The new attack costs almost a third of its detections.** The model that never saw Fuzzers catches 69% of
them instead of 99%: about 1,560 more fuzzing flows get through. Because Fuzzers is a large share of all
attacks, overall recall falls from 0.99 to 0.89. The model did not get worse at what it knew: recall on
the other eight attack types barely moves (0.995 to 0.984). As the brief puts it, the world moved and the
model stayed still.

**Why it still catches 69%.** Fuzzers share traits with the attacks the model has seen, above all the
attacker machines (D1): 93% of the Fuzzers test flows carry the attacker fingerprint. But the fingerprint
alone is not enough: the hidden model catches only 67% of the Fuzzers *with* the fingerprint. It has
learned the fingerprint *together with* the behaviour of the known attacks, and fuzzing behaves
differently, so many fingerprinted Fuzzers flows look benign to it. (The 343 Fuzzers flows without the
fingerprint are caught at 91%, but they are too few to read much into.)

**The dangerous part: the dashboard looks better.** With the hidden attack, the false alarm rate *falls*
(0.028 to 0.019), because the model now calls fewer flows attacks. Nothing in the alert count signals that
a new attack is getting through. A team watching only the number of alerts would see fewer alarms and
think things had improved.

**But the model is visibly less sure.** For 82% of the Fuzzers flows the hidden model's attack probability
lies between 0.1 and 0.9, against 43% for the model that saw them. A rising share of uncertain predictions
is a signal that does not need labels, and step G4 uses it.